In [ ]:
'''
A comparison of Tuned and Non Tuned Models: 
Models of Comparison, is one statistical model and one ML model

Statistical: Prophet 
ML: Random Forest Regressor

'''

In [1]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np
import pandas as pd 
import polars as pl
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df_Prophet_NonTuned = pd.read_parquet('prophet_non_tuned_forecast.parquet')
df_Prophet_Tuned = pd.read_parquet('prophet_tuned_forecast.parquet')

In [3]:
len(df_Prophet_Tuned) == len(df_Prophet_NonTuned)

True

In [4]:
df_RF_NonTuned = pd.read_parquet('randomforest_non_tuned_forecast.parquet')
df_RF_Tuned = pd.read_parquet('randomforest_tuned_forecast.parquet')

In [5]:
len(df_RF_NonTuned) == (len(df_RF_Tuned))

True

In [6]:
df_RF_Tuned.columns

Index(['unique_id', 'ds', 'cutoff', 'y', 'RandomForestRegressor'], dtype='object')

In [7]:
df_Prophet_Tuned = df_Prophet_Tuned.rename(columns={'yhat':'Prophet Tuned'})
df_Prophet_NonTuned = df_Prophet_NonTuned.rename(columns={'yhat':'Prophet Non-Tuned'})

df_RF_Tuned = df_RF_Tuned.rename(columns={'RandomForestRegressor':'RF Tuned'})
df_RF_NonTuned = df_RF_NonTuned.rename(columns={'RandomForestRegressor':'RF Non-Tuned'})

In [8]:
cutoffs = sorted(df_RF_Tuned['cutoff'].unique().tolist())
cutoffs

[Timestamp('2013-09-01 00:00:00'),
 Timestamp('2017-09-01 00:00:00'),
 Timestamp('2021-09-01 00:00:00')]

In [9]:
cutoff_dict = {
    cutoffs[0]: 'Validation 1',
    cutoffs[1]: 'Validation 2',
    cutoffs[2]: 'Test'
}

In [10]:
df_RF_All = df_RF_Tuned.merge(df_RF_NonTuned[['unique_id','ds','RF Non-Tuned']], on=['unique_id','ds'])
df_RF_All

,unique_id,ds,cutoff,y,RF Tuned,RF Non-Tuned
0,1,2013-10-01,2013-09-01,16.8,18.605669,18.687
1,1,2013-11-01,2013-09-01,17.8,18.130850,18.033
2,1,2013-12-01,2013-09-01,17.3,17.679738,17.680
3,1,2014-01-01,2013-09-01,16.3,17.357871,17.814
4,1,2014-02-01,2013-09-01,17.7,17.787660,17.827
...,...,...,...,...,...,...
79771,554,2025-05-01,2021-09-01,18.9,19.578819,19.535
79772,554,2025-06-01,2021-09-01,21.8,19.859530,19.814
79773,554,2025-07-01,2021-09-01,22.3,19.931009,19.618
79774,554,2025-08-01,2021-09-01,24.5,19.935700,19.673


In [11]:
df_Prophet_Tuned_Reduced = df_Prophet_Tuned[["unique_id","ds","set","y","Prophet Tuned"]]
df_Prophet_Tuned_Reduced

,unique_id,ds,set,y,Prophet Tuned
0,1,2013-10-01,Validation 1,16.8,19.081428
1,1,2013-11-01,Validation 1,17.8,18.730590
2,1,2013-12-01,Validation 1,17.3,19.080388
3,1,2014-01-01,Validation 1,16.3,18.603691
4,1,2014-02-01,Validation 1,17.7,18.965971
...,...,...,...,...,...
79771,554,2025-05-01,Test,18.9,17.446229
79772,554,2025-06-01,Test,21.8,16.692002
79773,554,2025-07-01,Test,22.3,16.566974
79774,554,2025-08-01,Test,24.5,16.917709


In [12]:
df_Prophet_All = df_Prophet_Tuned_Reduced.merge(df_Prophet_NonTuned[['unique_id','ds','set','Prophet Non-Tuned']], on=['unique_id','ds','set'], how='left')
df_Prophet_All.head()

,unique_id,ds,set,y,Prophet Tuned,Prophet Non-Tuned
0,1,2013-10-01,Validation 1,16.8,19.081428,19.156490
1,1,2013-11-01,Validation 1,17.8,18.730590,18.798675
2,1,2013-12-01,Validation 1,17.3,19.080388,19.138548
3,1,2014-01-01,Validation 1,16.3,18.603691,18.775863
4,1,2014-02-01,Validation 1,17.7,18.965971,19.046534


In [13]:
df_RF_All['set'] = df_RF_All['cutoff'].map(cutoff_dict)
df_RF_All.head()

,unique_id,ds,cutoff,y,RF Tuned,RF Non-Tuned,set
0,1,2013-10-01,2013-09-01,16.8,18.605669,18.687,Validation 1
1,1,2013-11-01,2013-09-01,17.8,18.130850,18.033,Validation 1
2,1,2013-12-01,2013-09-01,17.3,17.679738,17.680,Validation 1
3,1,2014-01-01,2013-09-01,16.3,17.357871,17.814,Validation 1
4,1,2014-02-01,2013-09-01,17.7,17.787660,17.827,Validation 1


In [14]:
df_All = df_RF_All.merge(df_Prophet_All[['unique_id','ds',"Prophet Tuned","Prophet Non-Tuned",'set']], on=['unique_id','ds','set'], how='left')
df_All.head()

,unique_id,ds,cutoff,y,RF Tuned,RF Non-Tuned,set,Prophet Tuned,Prophet Non-Tuned
0,1,2013-10-01,2013-09-01,16.8,18.605669,18.687,Validation 1,19.081428,19.156490
1,1,2013-11-01,2013-09-01,17.8,18.130850,18.033,Validation 1,18.730590,18.798675
2,1,2013-12-01,2013-09-01,17.3,17.679738,17.680,Validation 1,19.080388,19.138548
3,1,2014-01-01,2013-09-01,16.3,17.357871,17.814,Validation 1,18.603691,18.775863
4,1,2014-02-01,2013-09-01,17.7,17.787660,17.827,Validation 1,18.965971,19.046534


In [15]:
df_All.tail(20)

,unique_id,ds,cutoff,y,RF Tuned,RF Non-Tuned,set,Prophet Tuned,Prophet Non-Tuned
79756,554,2024-02-01,2021-09-01,26.2,18.670678,18.849,Test,16.885884,17.794593
79757,554,2024-03-01,2021-09-01,28.0,18.695764,18.734,Test,17.924331,19.224297
79758,554,2024-04-01,2021-09-01,23.9,18.735082,18.927,Test,16.721562,17.767961
79759,554,2024-05-01,2021-09-01,23.1,18.760612,19.315,Test,16.897737,18.020530
79760,554,2024-06-01,2021-09-01,18.6,18.948108,19.727,Test,16.402701,17.540921
79761,554,2024-07-01,2021-09-01,19.2,19.291416,19.591,Test,16.274713,17.140186
79762,554,2024-08-01,2021-09-01,15.3,19.375417,19.493,Test,16.749917,18.090531
79763,554,2024-09-01,2021-09-01,16.2,19.312605,19.279,Test,15.406253,16.400296
79764,554,2024-10-01,2021-09-01,15.8,19.300193,19.172,Test,15.994373,17.276519
79765,554,2024-11-01,2021-09-01,15.9,19.256165,19.070,Test,16.358682,17.543111


In [16]:
sets = df_All['set'].unique().tolist()
models = ['Prophet Tuned','Prophet Non-Tuned','RF Tuned','RF Non-Tuned']
unique_ids = df_All['unique_id'].unique().tolist()

In [17]:
def smape(y_true, y_pred):
    return 100 * np.mean(np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [18]:
Metrics = []
for uid in unique_ids:
    Sample = df_All[df_All['unique_id']==uid]

    for Set in sets:

        SampleReduced = Sample[Sample['set']==Set]

        for model in models:
            SampleReduced = SampleReduced.dropna(subset=['y',model])
            avgY = SampleReduced['y'].mean()
            rmse = root_mean_squared_error(SampleReduced['y'], SampleReduced[model])
            mae = mean_absolute_error(SampleReduced['y'], SampleReduced[model])
            sMAPE = smape(SampleReduced['y'], SampleReduced[model])
            Metrics.append({
            'unique_id':uid,
            'Model': model,
            'CV RMSE': 100 * (rmse/avgY),
            'CV MAE': 100 * (mae/avgY),
            'sMAPE': sMAPE,
            'Set': Set,
            'r2': r2_score(SampleReduced['y'], SampleReduced[model])
            
        })

df_Metrics = pd.DataFrame(Metrics)
df_Metrics

,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set,r2
0,1,Prophet Tuned,13.622261,12.437495,11.820859,Validation 1,-2.341303
1,1,Prophet Non-Tuned,13.959599,12.820864,12.174224,Validation 1,-2.508838
2,1,RF Tuned,7.815064,6.234152,6.138541,Validation 1,-0.099721
3,1,RF Non-Tuned,8.024052,6.055352,5.969374,Validation 1,-0.159324
4,1,Prophet Tuned,8.355754,6.917700,6.951670,Validation 2,0.134754
...,...,...,...,...,...,...,...
6643,554,RF Non-Tuned,22.978991,18.477864,18.574698,Validation 2,0.068768
6644,554,Prophet Tuned,24.275952,17.839679,17.644095,Test,-0.260797
6645,554,Prophet Non-Tuned,21.878017,16.663666,16.385428,Test,-0.024020
6646,554,RF Tuned,20.970344,17.499018,17.299644,Test,0.059186


In [ ]:
#(554 Unique Series) x (4 Models) x (3 Validation Sets) = 6648 ✅

In [19]:
color_discrete_map = {
    'Prophet Non-Tuned': "#103C6B",
    'Prophet Tuned': "#165C48",
    'RF Non-Tuned': "#751D12",
    'RF Tuned': "#AC5428"
}

In [ ]:
Box = px.box(
    title='CV RMSE Distributions: Tuned Models vs. Non Tuned Models',
    data_frame=df_Metrics,
    x='Model',
    y='CV RMSE',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set',
    points=False,
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [21]:
Box.write_image('figures/CV_RMSE_Tuned_vs_Non_Tuned.svg')

In [107]:
Box = px.box(
    title='CV MAE Distributions: Tuned Models vs. Non Tuned Models',
    data_frame=df_Metrics,
    x='Model',
    y='CV MAE',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set',
    points=False,
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [ ]:
Box = px.box(
    title='R2 Distributions: Tuned Models vs. Non Tuned Models',
    data_frame=df_Metrics,
    x='Model',
    y='r2',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set',
    points=False,
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [122]:
Box.write_image('figures/r2_Tuned_vs_Non_Tuned.svg')

In [22]:
Box = px.box(
    title='sMAPE Distributions: Tuned Models vs. Non Tuned Models',
    data_frame=df_Metrics,
    x='Model',
    y='sMAPE',
    width=1500,
    height=900,
    color='Model',
    color_discrete_map=color_discrete_map,
    template='ggplot2',
    facet_col='Set',
    points=False,
)

Box.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=12),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4", 
)

Box.show()

In [23]:
Box.write_image('figures/sMAPE_Tuned_vs_Non_Tuned.svg')

In [105]:
df_CV_RMSE = df_Metrics.groupby('Model')['CV RMSE'].describe().round(3).reset_index()
df_SMAPE = df_Metrics.groupby('Model')['sMAPE'].describe().round(3).reset_index()
df_R2 = df_Metrics[df_Metrics['r2']>=0].groupby('Model')['r2'].describe().round(3).reset_index()

In [25]:
df_CV_RMSE

,Model,count,mean,std,min,25%,50%,75%,max
0,Prophet Non-Tuned,1662.0,18.528,10.278,0.713,11.034,16.301,23.916,83.781
1,Prophet Tuned,1662.0,18.842,10.730,1.227,11.313,16.511,24.016,89.454
2,RF Non-Tuned,1662.0,19.080,11.036,1.967,11.324,16.647,23.894,94.251
3,RF Tuned,1662.0,18.797,10.813,1.876,11.175,16.542,23.449,94.490


In [27]:
df_SMAPE

,Model,count,mean,std,min,25%,50%,75%,max
0,Prophet Non-Tuned,1662.0,15.574,9.114,0.569,8.910,13.502,19.980,87.657
1,Prophet Tuned,1662.0,15.866,9.885,1.050,9.212,13.689,20.023,122.185
2,RF Non-Tuned,1662.0,15.655,8.901,1.615,9.235,13.728,19.710,66.076
3,RF Tuned,1662.0,15.438,8.688,1.532,9.095,13.634,19.368,66.526


In [106]:
df_R2

,Model,count,mean,std,min,25%,50%,75%,max
0,Prophet Non-Tuned,426.0,0.257,0.229,0.000,0.075,0.174,0.382,0.940
1,Prophet Tuned,391.0,0.273,0.242,0.002,0.078,0.186,0.434,0.884
2,RF Non-Tuned,298.0,0.135,0.132,0.001,0.038,0.095,0.187,0.762
3,RF Tuned,335.0,0.131,0.131,0.000,0.040,0.088,0.187,0.691


In [28]:
df_CV_RMSE.to_csv('metrics_summaries/CV_RMSE_Tuning.csv',index=False)
df_SMAPE.to_csv('metrics_summaries/SMAPE_Tuning.csv', index=False)

Determining if Prophet is Significantly better than its Non-Tuned Counter Part and the RF modesl

In [80]:
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon

In [81]:
df_Metrics

,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set,r2
0,1,Prophet Tuned,13.622261,12.437495,11.820859,Validation 1,-2.341303
1,1,Prophet Non-Tuned,13.959599,12.820864,12.174224,Validation 1,-2.508838
2,1,RF Tuned,7.815064,6.234152,6.138541,Validation 1,-0.099721
3,1,RF Non-Tuned,8.024052,6.055352,5.969374,Validation 1,-0.159324
4,1,Prophet Tuned,8.355754,6.917700,6.951670,Validation 2,0.134754
...,...,...,...,...,...,...,...
6643,554,RF Non-Tuned,22.978991,18.477864,18.574698,Validation 2,0.068768
6644,554,Prophet Tuned,24.275952,17.839679,17.644095,Test,-0.260797
6645,554,Prophet Non-Tuned,21.878017,16.663666,16.385428,Test,-0.024020
6646,554,RF Tuned,20.970344,17.499018,17.299644,Test,0.059186


In [115]:
df_pivot = df_Metrics.pivot_table(
    index=['unique_id','Set'],
    columns='Model',
    values='CV RMSE',
).reset_index()

df_pivot

Model,unique_id,Set,Prophet Non-Tuned,Prophet Tuned,RF Non-Tuned,RF Tuned
0,1,Test,8.134520,7.207433,7.635987,6.688143
1,1,Validation 1,13.959599,13.622261,8.024052,7.815064
2,1,Validation 2,7.378820,8.355754,11.153805,9.426450
3,2,Test,28.379154,28.833706,30.909836,30.190433
4,2,Validation 1,22.738165,22.627329,21.859699,21.576636
...,...,...,...,...,...,...
1657,553,Validation 1,19.082848,24.178795,11.141832,10.345779
1658,553,Validation 2,9.884892,12.069957,14.910910,13.954939
1659,554,Test,21.878017,24.275952,20.770371,20.970344
1660,554,Validation 1,30.427170,29.605082,29.698214,29.818865


In [116]:
ModelsLeft = ['Prophet Non-Tuned', 'RF Tuned', 'RF Non-Tuned']


In [117]:
Sets = df_pivot['Set'].unique()

In [118]:
test = []
for set in Sets:
    for model in ModelsLeft:
        df_pivot_reduced = df_pivot[df_pivot['Set']==set]

        t_stat, p_val = ttest_rel(df_pivot_reduced['Prophet Tuned'], df_pivot_reduced[model], alternative='two-sided')
    
        wilcoxon_result = wilcoxon(
        df_pivot_reduced['Prophet Tuned'],
        df_pivot_reduced[model],
        alternative='two-sided'
        )
    
        test.append({
            'Model':model,
            'Set': set,
            't-statistic':t_stat,
            't-test p': p_val,
            'Wilcoxon p': wilcoxon_result.pvalue
        })

test

[{'Model': 'Prophet Non-Tuned',
  'Set': 'Test',
  't-statistic': 0.6885396869975715,
  't-test p': 0.49140173122440534,
  'Wilcoxon p': 0.5740890157991514},
 {'Model': 'RF Tuned',
  'Set': 'Test',
  't-statistic': 1.3818131886972425,
  't-test p': 0.16758701525927497,
  'Wilcoxon p': 0.08144500538954454},
 {'Model': 'RF Non-Tuned',
  'Set': 'Test',
  't-statistic': 0.8208353476163991,
  't-test p': 0.4120937578198878,
  'Wilcoxon p': 0.3662702984454228},
 {'Model': 'Prophet Non-Tuned',
  'Set': 'Validation 1',
  't-statistic': 0.34525109122611813,
  't-test p': 0.7300368941441052,
  'Wilcoxon p': 0.26865231016874824},
 {'Model': 'RF Tuned',
  'Set': 'Validation 1',
  't-statistic': -1.5119843399222488,
  't-test p': 0.13110907724255674,
  'Wilcoxon p': 0.04736849760493921},
 {'Model': 'RF Non-Tuned',
  'Set': 'Validation 1',
  't-statistic': -2.3784204575024877,
  't-test p': 0.01772563218640079,
  'Wilcoxon p': 0.002560728271162094},
 {'Model': 'Prophet Non-Tuned',
  'Set': 'Validati

In [119]:
df_Hypothesis_Tests = pd.DataFrame(test)
df_Hypothesis_Tests

,Model,Set,t-statistic,t-test p,Wilcoxon p
0,Prophet Non-Tuned,Test,0.688540,0.491402,0.574089
1,RF Tuned,Test,1.381813,0.167587,0.081445
2,RF Non-Tuned,Test,0.820835,0.412094,0.366270
3,Prophet Non-Tuned,Validation 1,0.345251,0.730037,0.268652
4,RF Tuned,Validation 1,-1.511984,0.131109,0.047368
5,RF Non-Tuned,Validation 1,-2.378420,0.017726,0.002561
6,Prophet Non-Tuned,Validation 2,3.655355,0.000281,0.015242
7,RF Tuned,Validation 2,0.738407,0.460580,0.839689
8,RF Non-Tuned,Validation 2,-0.702062,0.482936,0.047606


In [120]:
df_Hypothesis_Tests.to_csv('metrics_summaries/Hypothesis_Tests.csv',index=False)

In [96]:
df_pivot['Prophet Tuned vs. Prophet Non-Tuned'] = df_pivot['Prophet Tuned'] - df_pivot['Prophet Non-Tuned']
df_pivot['Prophet Tuned vs. RF Non-Tuned'] = df_pivot['Prophet Tuned'] - df_pivot['RF Non-Tuned']
df_pivot['Prophet Tuned vs. RF Tuned'] = df_pivot['Prophet Tuned'] - df_pivot['RF Tuned']
df_pivot

Model,unique_id,Set,Prophet Non-Tuned,Prophet Tuned,RF Non-Tuned,RF Tuned,Prophet Tuned vs. Prophet Non-Tuned,Prophet Tuned vs. RF Non-Tuned,Prophet Tuned vs. RF Tuned
0,1,Test,8.134520,7.207433,7.635987,6.688143,-0.927087,-0.428554,0.519290
1,1,Validation 1,13.959599,13.622261,8.024052,7.815064,-0.337337,5.598209,5.807198
2,1,Validation 2,7.378820,8.355754,11.153805,9.426450,0.976933,-2.798051,-1.070697
3,2,Test,28.379154,28.833706,30.909836,30.190433,0.454551,-2.076130,-1.356728
4,2,Validation 1,22.738165,22.627329,21.859699,21.576636,-0.110836,0.767630,1.050693
...,...,...,...,...,...,...,...,...,...
1657,553,Validation 1,19.082848,24.178795,11.141832,10.345779,5.095947,13.036962,13.833016
1658,553,Validation 2,9.884892,12.069957,14.910910,13.954939,2.185065,-2.840952,-1.884981
1659,554,Test,21.878017,24.275952,20.770371,20.970344,2.397936,3.505582,3.305609
1660,554,Validation 1,30.427170,29.605082,29.698214,29.818865,-0.822088,-0.093132,-0.213783


In [97]:
Difference_Distribution = px.histogram(
    title='Prophet Tuned (CV RMSE) - Prophet Non-Tuned (CV RMSE)',
    data_frame=df_pivot,
    x='Prophet Tuned vs. Prophet Non-Tuned',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    marginal='box',
    histnorm='percent',
    
)

# Change bar color and add borders
Difference_Distribution.update_traces(
    marker_color="#2F56C4",     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Proportion of Series',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution.show()

In [39]:
Difference_Distribution.write_image('figures/CV_RMSE_ProphetT_vs_ProphetNonT.svg')

In [89]:
Difference_Distribution_Cumulative = px.histogram(
    title='Prophet Tuned (CV RMSE) - Prophet Non-Tuned (CV RMSE) - CDF',
    data_frame=df_pivot,
    x='Prophet Tuned vs. Prophet Non-Tuned',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    histnorm='percent',
    cumulative=True,
    
)

# Change bar color and add borders
Difference_Distribution_Cumulative.update_traces(
    marker_color="#2F56C4",     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution_Cumulative.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.add_hline(
    y=80,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Cumulative Proportion',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution_Cumulative.show()

In [41]:
Difference_Distribution_Cumulative.write_image('figures/CV_RMSE_ProphetT_vs_ProphetNonT_CDF.svg')

In [42]:
Difference_Distribution = px.histogram(
    title='Prophet Tuned (CV RMSE) - RF Non-Tuned (CV RMSE)',
    data_frame=df_pivot,
    x='Prophet Tuned vs. RF Non-Tuned',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    marginal='box',
    histnorm='percent',
    
)

# Change bar color and add borders
Difference_Distribution.update_traces(
    marker_color="#8E4C20",     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Proportion of Series',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution.show()

In [43]:
Difference_Distribution.write_image('figures/CV_RMSE_ProphetT_vs_RFNT.svg')

In [44]:
Difference_Distribution_Cumulative = px.histogram(
    title='Prophet Tuned (CV RMSE) - RF Non-Tuned (CV RMSE) - CDF',
    data_frame=df_pivot,
    x='Prophet Tuned vs. RF Non-Tuned',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    histnorm='percent',
    cumulative=True,
    
)

# Change bar color and add borders
Difference_Distribution_Cumulative.update_traces(
    marker_color="#8E4C20",     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution_Cumulative.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.add_hline(
    y=80,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Cumulative Proportion',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution_Cumulative.show()

In [45]:
Difference_Distribution_Cumulative.write_image('figures/CV_RMSE_ProphetT_vs_RFNT_CDF.svg')

In [46]:
Difference_Distribution = px.histogram(
    title='Prophet Tuned (CV RMSE) - RF Tuned (CV RMSE)',
    data_frame=df_pivot,
    x='Prophet Tuned vs. RF Tuned',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    marginal='box',
    histnorm='percent',
    
)

# Change bar color and add borders
Difference_Distribution.update_traces(
    marker_color="#15744E",     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Proportion of Series',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution.show()

In [47]:
Difference_Distribution.write_image('figures/CV_RMSE_ProphetT_vs_RFT.svg')

In [48]:
Difference_Distribution_Cumulative = px.histogram(
    title='Prophet Tuned (CV RMSE) - RF Tuned (CV RMSE) - CDF',
    data_frame=df_pivot,
    x='Prophet Tuned vs. RF Tuned',
    nbins=50,
    width=1500,
    height=900,
    facet_col='Set',
    histnorm='percent',
    cumulative=True,
    
)

# Change bar color and add borders
Difference_Distribution_Cumulative.update_traces(
    marker_color="#15744E",     # histogram color
    marker_line_color='black',  # border color
    marker_line_width=1         # border width
)

# Add vertical line at x = 0
Difference_Distribution_Cumulative.add_vline(
    x=0,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.add_hline(
    y=80,
    line_color='red',
    line_width=2,
    line_dash='dash'
)

Difference_Distribution_Cumulative.update_layout(
     title_font=dict(size=16),  # Set the title font size
    font=dict(size=15),  # Set general font size for labels and axis titles
            hoverlabel=dict(
                font_size=20  # Set font size for hover data
            ),
            yaxis=dict(
                title='Cumulative Proportion',
                title_font=dict(size=18),  # Set y-axis title font size
                tickfont=dict(size=12)  # Set y-axis tick font size
            ),
             legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.3,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#B4C2D4", 
)

Difference_Distribution_Cumulative.show()

In [49]:
Difference_Distribution_Cumulative.write_image('figures/CV_RMSE_ProphetT_vs_RFT_CDF.svg')

In [50]:
df_Prophet_Tuned_Reduced = df_Prophet_Tuned[["unique_id","ds","cutoff","set","y","Prophet Tuned"]]

In [51]:
cutoffs_dict = df_Prophet_Tuned_Reduced.set_index('cutoff')['set'].to_dict()

In [52]:
df_Metrics

,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set,r2
0,1,Prophet Tuned,13.622261,12.437495,11.820859,Validation 1,-2.341303
1,1,Prophet Non-Tuned,13.959599,12.820864,12.174224,Validation 1,-2.508838
2,1,RF Tuned,7.815064,6.234152,6.138541,Validation 1,-0.099721
3,1,RF Non-Tuned,8.024052,6.055352,5.969374,Validation 1,-0.159324
4,1,Prophet Tuned,8.355754,6.917700,6.951670,Validation 2,0.134754
...,...,...,...,...,...,...,...
6643,554,RF Non-Tuned,22.978991,18.477864,18.574698,Validation 2,0.068768
6644,554,Prophet Tuned,24.275952,17.839679,17.644095,Test,-0.260797
6645,554,Prophet Non-Tuned,21.878017,16.663666,16.385428,Test,-0.024020
6646,554,RF Tuned,20.970344,17.499018,17.299644,Test,0.059186


In [53]:
df_Prophet_Metrics = df_Metrics[df_Metrics['Model'] == 'Prophet Tuned']
df_Prophet_Metrics = df_Prophet_Metrics.sort_values(by='r2', ascending=False)

Residuals Analysis

In [54]:
def Plot_Residuals_Analysis(ts, unique_id):
    nlags = 48  #
    residuals = ts['Prophet_Residuals']
    acf_vals = acf(residuals, nlags=nlags)

    # Create subplot layout: 2 rows, 2 columns for row 2
    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=False,
        column_widths=[0.5, 0.5],
        row_heights=[0.3, 0.3, 0.30],  # first row taller
        specs=[[{"colspan": 2}, None],[{"colspan": 2}, None], [{"type": "xy"}, {"type": "xy"}]],
        subplot_titles=("Actual vs. Predicted","Residuals over Time", "Residuals Histogram", "Residuals ACF"),
        vertical_spacing=0.10
    )

    fig.add_trace(
        go.Scatter(
            x=ts['ds'],
            y=ts['y'],
            name='Actual',

            mode='lines',
            line=dict(color="#C25960", width=2.5),
        ),
        row=1, col=1
    )

    for c, n in cutoff_dict.items():
        fig.add_vline(x=c, line_dash="dash", line_color="black", row=1, col=1)
        

    fig.add_trace(
        go.Scatter(
            x=ts['ds'],
            y=ts['Prophet Tuned'],
            name='Predicted',
            mode='lines',
            line=dict(color=color_discrete_map['Prophet Tuned'], width=2.5, dash='dash'),
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=ts['ds'],
            y=residuals,
            mode='lines',
            line=dict(color="#221B1B", width=2.5),
            name='Residuals',
            showlegend=False 
        ),
        row=2, col=1
    )

    fig.add_hline(y=0, line_dash="dash", line_color="black", row=2, col=1)

    fig.add_trace(
        go.Histogram(
            x=residuals,
            nbinsx=30,
            name='Residuals',
            marker_color="#221B1B",
            marker_line_color="#DDD6D6",   # border color
            marker_line_width=1          # border width
        ),
        row=3, col=1
    )

    fig.add_trace(
        go.Bar(
            x=np.arange(len(acf_vals)),
            y=acf_vals,
            name='ACF',
            marker_color="#221B1B",
            showlegend=False 
        ),
        row=3, col=2
    )
    conf_int = 1.96/np.sqrt(len(residuals))
    fig.add_hline(y=conf_int, line_dash="dot", line_color="red", row=3, col=2)
    fig.add_hline(y=-conf_int, line_dash="dot", line_color="red", row=3, col=2)

    # Layout adjustments
    fig.update_layout(
        height=1000,
        width=1500,
        title_text=f"Prophet Residual Analysis for Series: {unique_id}",
         legend=dict(
                orientation='h',       
                yanchor='bottom',     
                y=-0.1,               
                xanchor='center',      
                 x=0.5,                 
                font=dict(size=16)     
            ),
            plot_bgcolor="#CDD0D4"
        

    )

    return fig

In [135]:
df_Prophet_Metrics.sort_values(by=['sMAPE'], ascending=True)

,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set,r2
864,73,Prophet Tuned,1.227305,1.050497,1.049917,Validation 1,0.780079
868,73,Prophet Tuned,2.606198,1.506945,1.514594,Validation 2,0.435739
4992,417,Prophet Tuned,1.915232,1.610371,1.592756,Validation 1,0.780704
4972,415,Prophet Tuned,2.385702,1.707875,1.720774,Validation 2,0.409728
2004,168,Prophet Tuned,2.099600,1.803642,1.810878,Validation 1,0.481143
...,...,...,...,...,...,...,...
2420,202,Prophet Tuned,65.630519,58.891153,82.819618,Test,-7.761754
4352,363,Prophet Tuned,67.719477,60.327101,90.185839,Test,-4.494413
6040,504,Prophet Tuned,75.519367,63.936903,91.637338,Validation 2,-2.207607
5188,433,Prophet Tuned,73.566362,65.542861,95.287937,Validation 2,-3.526021


In [136]:
df_Prophet_Metrics[(df_Prophet_Metrics['r2'] >= 0.50) & (df_Prophet_Metrics['r2'] < 0.70) & (df_Prophet_Metrics['Set']=='Test')]

,unique_id,Model,CV RMSE,CV MAE,sMAPE,Set,r2
5300,442,Prophet Tuned,5.147593,4.043042,4.045322,Test,0.683737
3128,261,Prophet Tuned,17.637953,14.498416,14.436320,Test,0.670472
2372,198,Prophet Tuned,11.916907,9.963633,10.227100,Test,0.665143
2564,214,Prophet Tuned,8.852872,7.158253,6.995599,Test,0.627186
3560,297,Prophet Tuned,19.391502,14.841830,15.557105,Test,0.600676
6596,550,Prophet Tuned,20.879115,15.676872,16.245226,Test,0.586073
5144,429,Prophet Tuned,3.323489,2.648386,2.696475,Test,0.584580
1616,135,Prophet Tuned,20.409565,15.387952,16.111705,Test,0.578930
3020,252,Prophet Tuned,2.524981,2.095150,2.109826,Test,0.570419
4964,414,Prophet Tuned,15.399562,11.905253,12.354214,Test,0.551549


In [125]:
df_Prophet_Tuned_Reduced

,unique_id,ds,cutoff,set,y,Prophet Tuned
0,1,2013-10-01,2013-09-01,Validation 1,16.8,19.081428
1,1,2013-11-01,2013-09-01,Validation 1,17.8,18.730590
2,1,2013-12-01,2013-09-01,Validation 1,17.3,19.080388
3,1,2014-01-01,2013-09-01,Validation 1,16.3,18.603691
4,1,2014-02-01,2013-09-01,Validation 1,17.7,18.965971
...,...,...,...,...,...,...
79771,554,2025-05-01,2021-09-01,Test,18.9,17.446229
79772,554,2025-06-01,2021-09-01,Test,21.8,16.692002
79773,554,2025-07-01,2021-09-01,Test,22.3,16.566974
79774,554,2025-08-01,2021-09-01,Test,24.5,16.917709


In [138]:
unique_id = 297 
ts = df_Prophet_Tuned_Reduced[df_Prophet_Tuned_Reduced['unique_id']==unique_id]
ts = ts.sort_values(by='ds')
ts = ts.drop_duplicates(subset=['ds'], keep='last')

ts['Prophet_Residuals'] = ts['Prophet Tuned'] - ts['y']

Residuals_297 = Plot_Residuals_Analysis(ts, unique_id)

In [139]:
Residuals_297

In [140]:
Residuals_297.write_image(f'figures/Residuals_Analysis_{unique_id}.svg')

In [133]:
unique_id = 433
ts = df_Prophet_Tuned_Reduced[df_Prophet_Tuned_Reduced['unique_id']==unique_id]
ts = ts.sort_values(by='ds')
ts = ts.drop_duplicates(subset=['ds'], keep='last')

ts['Prophet_Residuals'] = ts['Prophet Tuned'] - ts['y']

Residuals_433 = Plot_Residuals_Analysis(ts, unique_id)

In [134]:
Residuals_433

In [132]:
Residuals_73.write_image(f'figures/Residuals_Analysis_{unique_id}_badperformance.svg')

In [70]:
unique_id = 221
ts = df_Prophet_Tuned_Reduced[df_Prophet_Tuned_Reduced['unique_id']==unique_id]
ts = ts.sort_values(by='ds')
ts = ts.drop_duplicates(subset=['ds'], keep='last')

ts['Prophet_Residuals'] = ts['Prophet Tuned'] - ts['y']

Residuals_221 = Plot_Residuals_Analysis(ts, unique_id)

In [71]:
Residuals_221

In [72]:
Residuals_221.write_image(f'figures/Residuals_Analysis_{unique_id}_moderateperofrmance.svg')